In [ ]:
import pandas as pd
import numpy as np
import flirt
import os
import jupyter
import ipywidgets

# 1. Load WESAD dataset

In [2]:
# load data
df_acc = pd.read_parquet('data-input/dataset_wesad_wrist_acc.parquet')
df_bvp = pd.read_parquet('data-input/dataset_wesad_wrist_bvp.parquet')
df_eda = pd.read_parquet('data-input/dataset_wesad_wrist_eda.parquet')
df_temp = pd.read_parquet('data-input/dataset_wesad_wrist_temp.parquet')

# 2. Function for getting features from FLIRT

In [3]:
def get_features_inner(df, columns_list, prefix, window_length, window_step_size, frequency):
     
    # we need to set a correct datetime index (nanoseconds calculated from frequency)
    # otherwise Flirt will create a wrong timeindex
    ns = '250000000N'
    if frequency == 64:
        ns = '15625000N'
    elif frequency == 32:
        ns = '31250000N'
    time_index = pd.date_range(start=0, periods=len(df), freq=ns)
    df = df.set_index(time_index)
    
    df = df[columns_list]
    df = df.dropna()

    features = flirt.get_acc_features(df,
                                      window_length=window_length, 
                                      window_step_size=window_step_size,
                                      data_frequency=frequency)
    features = features.add_prefix(prefix)
    return features

In [4]:
def get_features(subject, label, df_acc, df_bvp, df_eda, df_temp, window_length, window_step_size):
    
    # calculate features
    acc_features = get_features_inner(df_acc, ['x', 'y', 'z'], 'acc_', window_length, window_step_size, 32)
    bvp_features = get_features_inner(df_bvp, ['BVP'], 'bvp_', window_length, window_step_size, 64)
    eda_features = get_features_inner(df_eda, ['EDA'], 'eda_', window_length, window_step_size, 4)
    temp_features = get_features_inner(df_temp, ['TEMP'], 'temp_', window_length, window_step_size, 4)

    # merge
    res = pd.concat([bvp_features, acc_features, eda_features, temp_features], axis=1)
    
    # add subject and label column
    res['subject'] = subject
    res['label'] = label

    return res

# 3. Calculate features for the whole dataset

In [5]:
df_acc.shape

(882470, 6)

In [6]:
df_acc

,x,y,z,subject,label,session
8940,63.0,4.0,9.0,5,0,1
8941,62.0,4.0,9.0,5,0,1
8942,63.0,4.0,9.0,5,0,1
8943,62.0,3.0,10.0,5,0,1
8944,62.0,4.0,10.0,5,0,1
...,...,...,...,...,...,...
121123,62.0,-13.0,1.0,15,1,2
121124,62.0,-13.0,0.0,15,1,2
121125,62.0,-13.0,0.0,15,1,2
121126,61.0,-12.0,0.0,15,1,2


In [7]:
iterlist = [(i, j)
    for i in df_acc.subject.unique()
    for j in df_acc.label.unique()]

In [8]:
def get_chunks(df, subject, label):
    df_chunk = df[df['subject'] == subject]
    df_chunk = df_chunk[df_chunk['label'] == label]
    df_chunk = df_chunk.drop(columns=['session', 'subject', 'label'])
    return df_chunk

In [9]:
def get_all_chunks(df_acc, df_bvp, df_eda, df_temp, subject, label):
    df_acc_chunk = get_chunks(df_acc, subject, label)
    df_bvp_chunk = get_chunks(df_bvp, subject, label)
    df_eda_chunk = get_chunks(df_eda, subject, label)
    df_temp_chunk = get_chunks(df_temp, subject, label)
    return df_acc_chunk, df_bvp_chunk, df_eda_chunk, df_temp_chunk

In [12]:
%%time

result_dfs = []

window_length = 60
window_step_size = 10

# loop over all subject-label combinations (all subjects have 2 sessions with 1 label each)
for (subject, label) in iterlist:
    
    df_acc_chunk, df_bvp_chunk, df_eda_chunk, df_temp_chunk = get_all_chunks(df_acc, df_bvp, df_eda, df_temp, subject, label)
    
    res_df_chunks = get_features(subject, label, df_acc_chunk, df_bvp_chunk, df_eda_chunk, df_temp_chunk, window_length, window_step_size)
    result_dfs.append(res_df_chunks)

res = pd.concat(result_dfs)

/var/folders/kw/lny4507d14g5tp6x1qdzhh8r0000gn/T/ipykernel_14904/637032015.py:10: FutureWarning: 'N' is deprecated and will be removed in a future version, please use 'ns' instead.
  time_index = pd.date_range(start=0, periods=len(df), freq=ns)


ACC features:   0%|          | 0/120 [00:00<?, ?it/s]

/Users/ashidudissanayake/Dev/Shadow/.venv/lib/python3.11/site-packages/flirt/stats/common.py:35: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  results[key] = value(data)
/var/folders/kw/lny4507d14g5tp6x1qdzhh8r0000gn/T/ipykernel_14904/637032015.py:10: FutureWarning: 'N' is deprecated and will be removed in a future version, please use 'ns' instead.
  time_index = pd.date_range(start=0, periods=len(df), freq=ns)


ACC features:   0%|          | 0/120 [00:00<?, ?it/s]

/var/folders/kw/lny4507d14g5tp6x1qdzhh8r0000gn/T/ipykernel_14904/637032015.py:10: FutureWarning: 'N' is deprecated and will be removed in a future version, please use 'ns' instead.
  time_index = pd.date_range(start=0, periods=len(df), freq=ns)


...

/var/folders/kw/lny4507d14g5tp6x1qdzhh8r0000gn/T/ipykernel_14904/637032015.py:10: FutureWarning: 'N' is deprecated and will be removed in a future version, please use 'ns' instead.
  time_index = pd.date_range(start=0, periods=len(df), freq=ns)


ACC features:   0%|          | 0/69 [00:00<?, ?it/s]

/var/folders/kw/lny4507d14g5tp6x1qdzhh8r0000gn/T/ipykernel_14904/637032015.py:10: FutureWarning: 'N' is deprecated and will be removed in a future version, please use 'ns' instead.
  time_index = pd.date_range(start=0, periods=len(df), freq=ns)


ACC features:   0%|          | 0/69 [00:00<?, ?it/s]

CPU times: user 4.68 s, sys: 1.08 s, total: 5.76 s
Wall time: 14.7 s


In [13]:
res

,bvp_BVP_mean,bvp_BVP_std,bvp_BVP_min,bvp_BVP_max,bvp_BVP_ptp,bvp_BVP_sum,bvp_BVP_energy,bvp_BVP_skewness,bvp_BVP_kurtosis,bvp_BVP_peaks,...,temp_l2_n_sign_changes,temp_l2_iqr,temp_l2_iqr_5_95,temp_l2_pct_5,temp_l2_pct_95,temp_l2_entropy,temp_l2_perm_entropy,temp_l2_svd_entropy,subject,label
datetime,,,,,,,,,,,,,,,,,,,,,
1970-01-01 00:01:00,-0.336763,204.677350,-1092.57,743.27,1835.84,-1293.17,1.608689e+08,-0.656369,2.830672,144.0,...,0,0.36,0.5500,33.7900,34.340,5.480623,0.966334,0.004160,5,0
1970-01-01 00:01:10,-0.196458,206.018759,-1092.57,743.27,1835.84,-754.40,1.629841e+08,-0.655848,2.875779,143.0,...,0,0.26,0.5200,33.7500,34.270,5.480627,0.973835,0.004250,5,0
1970-01-01 00:01:20,-0.394151,193.980257,-1092.57,743.27,1835.84,-1513.54,1.444934e+08,-0.768833,3.984683,149.0,...,0,0.16,0.3825,33.7275,34.110,5.480633,0.926504,0.004224,5,0
1970-01-01 00:01:30,-0.383578,187.887658,-1092.57,743.27,1835.84,-1472.94,1.355594e+08,-0.732419,4.665437,150.0,...,0,0.12,0.2225,33.7275,33.950,5.480636,0.900485,0.004299,5,0
1970-01-01 00:01:40,-0.525495,187.737796,-1092.57,743.27,1835.84,-2017.90,1.353437e+08,-0.757962,4.586562,153.0,...,0,0.07,0.1635,33.7275,33.891,5.480637,0.853373,0.004196,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1970-01-01 00:11:40,-0.110550,33.987350,-194.79,215.27,410.06,-325.46,3.400768e+06,-0.027630,14.452518,71.0,...,0,0.02,0.0600,29.9700,30.030,5.209486,0.732901,0.003615,15,1
1970-01-01 00:11:50,-0.125846,33.997729,-194.79,191.77,386.56,-289.95,2.663105e+06,-0.518064,13.727508,56.0,...,0,0.02,0.0400,29.9700,30.010,4.962845,0.707410,0.003644,15,1
1970-01-01 00:12:00,0.270198,13.619682,-36.34,61.71,98.05,449.61,3.087864e+05,0.262036,1.429592,40.0,...,0,0.00,0.0400,29.9700,30.010,4.634729,0.634310,0.003034,15,1


In [14]:

# handle NANs
res = res.dropna()

In [15]:
res

,bvp_BVP_mean,bvp_BVP_std,bvp_BVP_min,bvp_BVP_max,bvp_BVP_ptp,bvp_BVP_sum,bvp_BVP_energy,bvp_BVP_skewness,bvp_BVP_kurtosis,bvp_BVP_peaks,...,temp_l2_n_sign_changes,temp_l2_iqr,temp_l2_iqr_5_95,temp_l2_pct_5,temp_l2_pct_95,temp_l2_entropy,temp_l2_perm_entropy,temp_l2_svd_entropy,subject,label
datetime,,,,,,,,,,,,,,,,,,,,,
1970-01-01 00:01:00,-0.336763,204.677350,-1092.57,743.27,1835.84,-1293.17,1.608689e+08,-0.656369,2.830672,144.0,...,0,0.36,0.5500,33.7900,34.340,5.480623,0.966334,0.004160,5,0
1970-01-01 00:01:10,-0.196458,206.018759,-1092.57,743.27,1835.84,-754.40,1.629841e+08,-0.655848,2.875779,143.0,...,0,0.26,0.5200,33.7500,34.270,5.480627,0.973835,0.004250,5,0
1970-01-01 00:01:20,-0.394151,193.980257,-1092.57,743.27,1835.84,-1513.54,1.444934e+08,-0.768833,3.984683,149.0,...,0,0.16,0.3825,33.7275,34.110,5.480633,0.926504,0.004224,5,0
1970-01-01 00:01:30,-0.383578,187.887658,-1092.57,743.27,1835.84,-1472.94,1.355594e+08,-0.732419,4.665437,150.0,...,0,0.12,0.2225,33.7275,33.950,5.480636,0.900485,0.004299,5,0
1970-01-01 00:01:40,-0.525495,187.737796,-1092.57,743.27,1835.84,-2017.90,1.353437e+08,-0.757962,4.586562,153.0,...,0,0.07,0.1635,33.7275,33.891,5.480637,0.853373,0.004196,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1970-01-01 00:11:40,-0.110550,33.987350,-194.79,215.27,410.06,-325.46,3.400768e+06,-0.027630,14.452518,71.0,...,0,0.02,0.0600,29.9700,30.030,5.209486,0.732901,0.003615,15,1
1970-01-01 00:11:50,-0.125846,33.997729,-194.79,191.77,386.56,-289.95,2.663105e+06,-0.518064,13.727508,56.0,...,0,0.02,0.0400,29.9700,30.010,4.962845,0.707410,0.003644,15,1
1970-01-01 00:12:00,0.270198,13.619682,-36.34,61.71,98.05,449.61,3.087864e+05,0.262036,1.429592,40.0,...,0,0.00,0.0400,29.9700,30.010,4.634729,0.634310,0.003034,15,1


In [16]:
# store as parquet

if not os.path.isdir('data-input'):
    os.makedirs('data-input')

res.to_parquet('data-input/flirt-wesad-acc-bvp-eda-temp-'+str(window_length)+'-'+str(window_step_size)+'.parquet')